# Uni-MuMER - Kaggle 2xT4 + DagsHub MLflow

Train Uni-MuMER trên Kaggle với:
- Miniconda → Python 3.10 → Train QLoRA 4-bit
- **DagsHub MLflow** tự động đồng bộ logs + checkpoint (~643MB) lên cloud
- Mặc định 2 GPU T4, tự động chuyển FP16 nếu detect T4

## 0. Tham số chính

In [1]:
import os
from kaggle_secrets import UserSecretsClient

YAML_CONFIG = "train/Uni-MuMER-train.yaml"
MODEL_DIR = "Uni-MuMER-Qwen2.5-VL-3B"

DAGSHUB_USERNAME = "NhatPot"
DAGSHUB_REPO = "test-unimer"
MLFLOW_TRACKING_URI = "https://dagshub.com/NhatPot/test-unimer.mlflow"

user_secrets = UserSecretsClient()
DAGSHUB_TOKEN = user_secrets.get_secret("DAGSHUB_TOKEN")

os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN

os.environ["DAGSHUB_TOKEN"] = DAGSHUB_TOKEN
os.environ["DAGSHUB_USERNAME"] = DAGSHUB_USERNAME
os.environ["DAGSHUB_REPO_NAME"] = DAGSHUB_REPO

print(f"MLflow URI: {MLFLOW_TRACKING_URI}")
print("DagsHub token loaded from Kaggle Secrets.")

MLflow URI: https://dagshub.com/NhatPot/test-unimer.mlflow
DagsHub token loaded from Kaggle Secrets.


## 1. Cài Miniconda

## 2. Tạo env Python 3.10

In [2]:
!rm -rf /kaggle/working/*
# Cài đặt Miniconda vào thư mục có quyền ghi
!wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!bash Miniconda3-latest-Linux-x86_64.sh -b -f -p /kaggle/working/miniconda
!rm Miniconda3-latest-Linux-x86_64.sh

# Đồng ý điều khoản của Anaconda (Bắt buộc để cài đặt các package từ channel main/r)
!/kaggle/working/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!/kaggle/working/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

# Tạo môi trường unimumer với Python 3.10
!/kaggle/working/miniconda/bin/conda create -n unimumer python=3.10 -y

--2026-06-21 15:47:27--  https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
Resolving repo.anaconda.com (repo.anaconda.com)... 104.16.32.241, 104.16.191.158, 2606:4700::6810:20f1, ...
Connecting to repo.anaconda.com (repo.anaconda.com)|104.16.32.241|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 163179296 (156M) [application/octet-stream]
Saving to: ‘Miniconda3-latest-Linux-x86_64.sh’

Miniconda3-latest-L 100%[===================>] 155.62M   258MB/s    in 0.6s    

2026-06-21 15:47:27 (258 MB/s) - ‘Miniconda3-latest-Linux-x86_64.sh’ saved [163179296/163179296]

PREFIX=/kaggle/working/miniconda
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best r

## 3. Clone repo

In [3]:
%cd /kaggle/working
!git clone https://github.com/NhatPot/test-unimer.git
%cd test-unimer
!pwd

/kaggle/working
Cloning into 'test-unimer'...
remote: Enumerating objects: 679, done.
remote: Counting objects: 100% (679/679), done.
remote: Compressing objects: 100% (450/450), done.
remote: Total 679 (delta 216), reused 670 (delta 207), pack-reused 0 (from 0)
Receiving objects: 100% (679/679), 14.91 MiB | 18.92 MiB/s, done.
Resolving deltas: 100% (216/216), done.
/kaggle/working/test-unimer
/kaggle/working/test-unimer


In [ ]:
# Define ACTIVATE command
ACTIVATE = "source /kaggle/working/miniconda/bin/activate unimumer"
print(f"ACTIVATE: {ACTIVATE}")

## 4. Setup DagsHub credentials

## 5. Download base model

In [4]:
!mkdir -p {MODEL_DIR}
!wget -q --show-progress -L -O {MODEL_DIR}/model-00001-of-00002.safetensors \
  "https://huggingface.co/phxember/Uni-MuMER-Qwen2.5-VL-3B/resolve/main/model-00001-of-00002.safetensors?download=true"
!wget -q --show-progress -L -O {MODEL_DIR}/model-00002-of-00002.safetensors \
  "https://huggingface.co/phxember/Uni-MuMER-Qwen2.5-VL-3B/resolve/main/model-00002-of-00002.safetensors?download=true"
!ls -lh {MODEL_DIR}/*.safetensors

Uni-MuMER-Qwen2.5-V 100%[===================>]   4.65G   108MB/s    in 21s     
Uni-MuMER-Qwen2.5-V 100%[===================>]   2.92G   294MB/s    in 9.9s    
-rw-r--r-- 1 root root 4.7G Jun 21 15:48 Uni-MuMER-Qwen2.5-VL-3B/model-00001-of-00002.safetensors
-rw-r--r-- 1 root root 3.0G Jun 21 15:48 Uni-MuMER-Qwen2.5-VL-3B/model-00002-of-00002.safetensors


## 6. Install dependencies

In [5]:
# --- BƯỚC 3: CÀI ĐẶT THƯ VIỆN ---
# Lưu ý: CUDA 12.1 hoặc 12.4 ổn định hơn trên Kaggle T4, bạn có thể thử 12.8 nếu cần
#!{ACTIVATE} && conda install pytorch torchvision torchaudio pytorch-cuda=12.1 -c pytorch -c nvidia -y
!{ACTIVATE} && pip install -r requirements.txt

# Cài đặt LLaMA-Factory
%cd train/LLaMA-Factory
!{ACTIVATE} && pip install -e .

# Quay lại thư mục gốc dự án
%cd ../..

/bin/bash: line 1: {ACTIVATE}: command not found
/kaggle/working/test-unimer/train/LLaMA-Factory
/bin/bash: line 1: {ACTIVATE}: command not found
/kaggle/working/test-unimer


## 8. Init DagsHub

In [8]:
# Verify dagshub installed in unimumer env
!source /kaggle/working/miniconda/bin/activate unimumer && python -c "import dagshub; print('✅ dagshub version:', dagshub.__version__)"

# Environment variables already set in cell 0
print(f"✅ MLflow URI: {os.environ['MLFLOW_TRACKING_URI']}")
print("✅ DagsHub will be initialized in training script")

/bin/bash: line 1: {ACTIVATE}: command not found
✅ DagsHub will be initialized in training script
   MLflow URI: https://dagshub.com/NhatPot/test-unimer.mlflow


## 9. Training

In [10]:
!source /kaggle/working/miniconda/bin/activate unimumer && \
 MPLBACKEND=Agg llamafactory-cli train train/Uni-MuMER-train.yaml


/bin/bash: line 1: llamafactory-cli: command not found


## 10. Upload to DagsHub

In [ ]:
from pathlib import Path
from datetime import datetime

# Load config
with open(YAML_CONFIG, 'r') as f:
    config = yaml.safe_load(f)

output_dir = config['output_dir']
checkpoints = sorted(Path(output_dir).glob("checkpoint-*"))
latest_checkpoint = str(checkpoints[-1]) if checkpoints else output_dir

# Generate run name
dataset = config.get('dataset', '')
if 'error' in str(dataset).lower():
    task = "EDL"
elif 'tree' in str(dataset).lower():
    task = "TreeCoT"
elif '_can' in str(dataset).lower():
    task = "SymbolCount"
else:
    task = "Train"

run_name = f"{task}_rank{config.get('lora_rank',64)}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

print(f"Output: {output_dir}")
print(f"Checkpoint: {latest_checkpoint}")
print(f"Run: {run_name}")
!du -sh {latest_checkpoint}

In [ ]:
# Upload tất cả lên DagsHub
!{ACTIVATE} && python scripts/post_training_logger.py \
  --yaml-config {YAML_CONFIG} \
  --output-dir {output_dir} \
  --checkpoint-dir {latest_checkpoint} \
  --run-name "{run_name}" \
  --dataset-info train/dataset_info.json

## 11. Verify & create backup

In [ ]:
# Check files
!echo "===== Important files ====="
!ls -lh {latest_checkpoint}/adapter_model.safetensors
!ls -lh {output_dir}/README_RUN.md
!ls -lh {output_dir}/ARTIFACT_MANIFEST.json

In [ ]:
# Create backup archive
archive = f"unimumer_{task}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.tar.gz"
!tar -czf /kaggle/working/{archive} -C {output_dir} .
!ls -lh /kaggle/working/{archive}

## 12. Summary

In [ ]:
print("="*60)
print("✅ TRAINING COMPLETE")
print("="*60)
print(f"\n📁 Local: {output_dir}")
print(f"📁 Checkpoint: {latest_checkpoint}")
print(f"📦 Archive: {archive}")
print(f"\n☁️  DagsHub: https://dagshub.com/{os.environ['DAGSHUB_USERNAME']}/{os.environ['DAGSHUB_REPO_NAME']}")
print(f"📊 Experiments: Click 'Experiments' tab → Find run '{run_name}'")
print(f"\n💾 Upload: Logs + Checkpoint (~643MB) uploaded to DagsHub")
print(f"📥 Download: Go to MLflow UI → Artifacts → Download")
print("="*60)